## Vessel segmentation using a U-Net model
This notebook demonstrates the use of the vseg vessel segmentation network included in the toolkit. The model is trained on LSFM data.

In [1]:
# Import the necessary libraries
import os
import shutil

import numpy as np
from liom_toolkit.segmentation.vseg.model import VsegModel
from liom_toolkit.segmentation.vseg.prediction import predict_one
from liom_toolkit.utils import extract_zarr_to_image, generate_label_color_dict_mask, save_label_to_zarr
from tqdm.auto import tqdm
import dask.array as da

### Prepare the zarr volume
The individual slices need to be extracted from the zarr volume and saved as individual png files. The following code demonstrates how to do this.

In [2]:
import os
# The full workflow extracts slices from an OME-Zarr volume via
# extract_zarr_to_image(zarr_file, png_dir, channel=0). Here we use the
# small set of slices bundled in data/vseg/S23_555nm_slices/ so the
# notebook runs without a 20+ GB lab zarr. To run on your own data,
# point zarr_file at your OME-Zarr volume and uncomment the extract call.
zarr_file = "data/vseg/output/vessels.zarr"
png_dir = "data/vseg/S23_555nm_slices"
os.makedirs(os.path.dirname(zarr_file), exist_ok=True)
# extract_zarr_to_image(zarr_file, png_dir, channel=0)  # uncomment for full volumes


### Run the model
The following code downloads the pre-trained model and runs it for the entire folder

In [3]:
# Set the device, 'cuda' if the machine has an Nvidia GPU and CUDA installed, otherwise 'cpu'
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
# Model load. pretrained=True downloads weights from a wandb model artifact;
# pretrained_artifact is the wandb registry path (requires `wandb login` +
# access to the liom-lab project). Use pretrained=False to train from scratch.
model = VsegModel(pretrained=True, pretrained_artifact="liom-lab/model-registry/Vessel Segmentation:latest", device=device)


In [4]:
dir_path = png_dir
output_path = "data/vseg/predictions"
os.makedirs(output_path, exist_ok=True)
normalization = True
patching = False
stride = None
width = None


In [5]:
# Run the model
vessel_stack = []
for images in tqdm(sorted(os.listdir(dir_path)), desc="Processing images"):
    if not images.endswith('.png'):
        continue
    image_path = os.path.join(dir_path, images)
    prediction = predict_one(model=model, img_path=image_path, save_path=output_path,
                             norm=normalization, dev=device, patching=patching)
    prediction_dask = da.from_array(prediction, chunks=(128, 128))
    vessel_stack.append(prediction_dask)
volume = da.stack(vessel_stack, axis=0)


Processing images:   0%|          | 0/3 [00:00<?, ?it/s]

### Reassemble the slices to save as zarr
Next, we need to save the files back to a zarr volume. The following code demonstrates how to do this.

In [6]:
# Make the volume binary
volume[volume > 0] = 1

# Remove any existing zarr store from a previous run before saving
import shutil, os
if os.path.exists(zarr_file):
    shutil.rmtree(zarr_file)

# Save the volume to a new OME-Zarr store
colour_dict = generate_label_color_dict_mask()
save_label_to_zarr(volume, zarr_file, colour_dict, "vessels", resolution_level=0)
print(f"Saved vessel predictions to {zarr_file}")


Saved vessel predictions to data/vseg/output/vessels.zarr


### Cleanup
Finally, we can clean up the png files that were created.

In [7]:
# Clean up the extracted PNG slices (only if you extracted them from a zarr
# above — the bundled slices are left in place for re-runs).
# shutil.rmtree(png_dir)
